In [ ]:
%pip install langchain langchain-community langchain-core langchain-text-splitters langchain-openai langchain-chroma pypdf chromadb datasets ragas

Note: you may need to restart the kernel to use updated packages.


In [11]:
%pip install matplotlib

  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   -------------- ------------------------- 2.9/8.2 MB 15.3 MB/s eta 0:00:01
   ------------------------------- -------- 6.6/8.2 MB 16.8 MB/s eta 0:00:01
   ---------------------------------------- 8.2/8.2 MB 15.9 MB/s  0:00:00
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 1.6/1.6 MB 16.8 MB/s  0:00:00

   ------------- -------------------------- 2/6 [fonttools]
   ------------- -------------------------- 2/6 [fonttools]
   ------------- -------------------------- 2/6 [fonttools]
   ------------- -------------------------- 2/6 [fonttools]
   ------------- -------------------------- 2/6 [fonttools]
   ------------- -------------------------- 2/6 [fonttools]
   ------------- -------------------------- 2/6 [fonttools]
   ------------- -----

In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

# LangChain & Ragas 패키지
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from operator import itemgetter
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy

# 1. 환경변수 로드
load_dotenv()

print("--- 1. 평가용 데이터셋 로드 ---")
# ⚠️ kora_eval_dataset.json 경로 설정 (실제 경로에 맞게 수정이 필요할 수 있습니다)
dataset_path = "../../docs/kora_eval_dataset.json"

try:
    with open(dataset_path, "r", encoding="utf-8") as f:
        eval_data = json.load(f)
    
    # JSON 구조에서 질문과 모범답안만 추출
    test_questions = [item['question'] for item in eval_data['dataset']]
    ground_truths = [item['ground_truth'] for item in eval_data['dataset']]
    print(f"총 {len(test_questions)}개의 테스트 세트가 준비되었습니다!")
except FileNotFoundError:
    print(f"⚠️ {dataset_path} 파일을 찾을 수 없습니다. 평가 데이터셋을 해당 위치에 먼저 준비해주세요.")
    test_questions = []
    ground_truths = []

c:\Users\didak\miniconda3\envs\inflearn-llm-application\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- 1. 평가용 데이터셋 로드 ---
⚠️ ../../docs/kora_eval_dataset.json 파일을 찾을 수 없습니다. 평가 데이터셋을 해당 위치에 먼저 준비해주세요.


C:\Users\didak\AppData\Local\Temp\ipykernel_17520\1088731467.py:17: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy
C:\Users\didak\AppData\Local\Temp\ipykernel_17520\1088731467.py:17: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy
C:\Users\didak\AppData\Local\Temp\ipykernel_17520\1088731467.py:17: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: 

In [ ]:
print("--- 2. 실험용 벡터 DB 구축 중 ---")
pdf_path = "../../docs/KORA 규정집.pdf"
loader = PyPDFLoader(pdf_path)
document = loader.load()

# 공통으로 사용할 임베딩 모델
embedding_model = OpenAIEmbeddings(model='text-embedding-3-large')

# 실험해볼 청크 사이즈 후보군
chunk_sizes_to_test = [250, 500, 750, 1000, 1250, 1500, 1750, 2000]
dbs = {}

for size in chunk_sizes_to_test:
    print(f"\n[{size} 사이즈] 청킹 및 DB 준비 중...")
    
    # 앞서 논의한 비용 절감을 위해 로컬 저장소 캐싱 로직 적용!
    persist_dir = f"./chroma_db_{size}"
    
    if os.path.exists(persist_dir):
        print(f" -> '{persist_dir}' 폴더가 존재하여 기존 로컬 DB를 로드합니다.")
        db = Chroma(persist_directory=persist_dir, embedding_function=embedding_model)
    else:
        print(f" -> '{persist_dir}' 새 DB를 생성하고 저장합니다...")
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=100)
        split_docs = text_splitter.split_documents(document)
        db = Chroma.from_documents(documents=split_docs, embedding=embedding_model, persist_directory=persist_dir)
    
    dbs[size] = db

print("\n✅ 모든 테스트용 DB 세팅 완료!")

In [ ]:
print("--- 3. Ragas 성능 평가 시작 (시간이 꽤 소요됩니다) ---")

if not test_questions:
    print("평가 데이터가 없어서 실행을 중단합니다. (Cell 1 확인)")
else:
    # RAG 생성용 모델 및 프롬프트
    llm = ChatOpenAI(model='gpt-4o', temperature=0)
    prompt = ChatPromptTemplate.from_template("""
    주어진 Context를 바탕으로 Question에 답하세요.
    Context: {context}
    Question: {question}
    Answer:
    """)

    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    evaluation_summaries = []

    # 각 청크 사이즈(DB)별로 실험 시작
    for chunk_size, db in dbs.items():
        print(f"\n▶️ 청크 사이즈 {chunk_size} 평가 진행 중...")
        
        retriever = db.as_retriever(search_kwargs={'k': 3})
        
        rag_chain = (
            {
                "context": itemgetter("question") | retriever | format_docs,
                "question": itemgetter("question")
            }
            | prompt
            | llm
            | StrOutputParser()
        )
        
        # 채점용 DTO 세팅
        data_for_ragas = {"question": [], "contexts": [], "answer": [], "ground_truth": []}
        
        # 질문을 순회하며 답변 생성
        for i, q in enumerate(test_questions):
            answer = rag_chain.invoke({"question": q})
            contexts = [doc.page_content for doc in retriever.invoke(q)]
            
            data_for_ragas["question"].append(q)
            data_for_ragas["contexts"].append(contexts)
            data_for_ragas["answer"].append(answer)
            data_for_ragas["ground_truth"].append(ground_truths[i])
            
        # Ragas 채점
        eval_dataset = Dataset.from_dict(data_for_ragas)
        score = evaluate(
            dataset=eval_dataset,
            metrics=[context_precision, context_recall, faithfulness, answer_relevancy],
        )
        
        # 결과를 저장
        score_dict = score.copy()
        score_dict['chunk_size'] = chunk_size
        evaluation_summaries.append(score_dict)

    print("\n🎉 모든 평가 완료!")

In [ ]:
# 결과를 DataFrame으로 변환 및 시각화
if 'evaluation_summaries' in locals() and evaluation_summaries:
    df_results = pd.DataFrame(evaluation_summaries)
    df_results.set_index('chunk_size', inplace=True)

    print("📊 [실험 결과 표]")
    display(df_results)

    # 한글 폰트 깨짐 방지 (윈도우 기준)
    plt.rcParams['font.family'] = 'Malgun Gothic'
    plt.rcParams['axes.unicode_minus'] = False

    # 시각화 (그래프 그리기)
    df_results.plot(kind='bar', figsize=(10, 6))
    plt.title('Chunk Size별 RAG 성능 지표 비교')
    plt.xlabel('Chunk Size')
    plt.ylabel('Score (0.0 ~ 1.0)')
    plt.ylim(0, 1.1)
    plt.legend(loc='lower right')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.xticks(rotation=0)
    plt.show()
else:
    print("시각화할 평가 결과가 없습니다.")